Environment Setup and Authentication

In [4]:
import sys
import os

# Upgrade Hugging Face ecosystem safely
!{sys.executable} -m pip install --upgrade --quiet transformers accelerate huggingface_hub datasets

import torch
import math
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login

# Hardware check
device_count = torch.cuda.device_count()
print(f"PyTorch {torch.__version__} | Detected {device_count} GPUs.")

# Authenticatation
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("Authenticated via Kaggle Secrets.")
except Exception as e:
    HF_TOKEN = "" 
    login(token=HF_TOKEN)
    print("Authenticated via explicit string.")

PyTorch 2.10.0+cu128 | Detected 2 GPUs.
Authenticated via explicit string.


Evaluation Metrices

In [5]:
class IntrinsicEvaluator:
    def __init__(self, model, tokenizer, device="cuda:0"):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        
    def get_lengths(self, text: str):
        """Calculates text-intrinsic properties for normalization."""
        if not isinstance(text, str):
            text = str(text)
        num_chars = max(1, len(text))
        num_bytes = max(1, len(text.encode('utf-8')))
        num_words = max(1, len(text.strip().split()))
        return num_chars, num_bytes, num_words

    def evaluate_sequence(self, text: str):
        """Calculates raw NLL and normalizes it into BPB, BPC, and BPW."""
        chars, bytes_count, words = self.get_lengths(text)
        inputs = self.tokenizer(text, return_tensors="pt").to(self.device)
        
        with torch.no_grad():
            outputs = self.model(**inputs, labels=inputs["input_ids"])
            # HF CausalLM loss is the average negative log-likelihood per token
            seq_len = inputs["input_ids"].shape[1]
            neg_log_likelihood = outputs.loss.item() * (seq_len - 1)
            
        # Calculate Tokenizer-Independent Metrics
        bpb = neg_log_likelihood / (bytes_count * math.log(2))
        bpc = neg_log_likelihood / (chars * math.log(2))
        bpw = neg_log_likelihood / (words * math.log(2))
        
        return {
            "nll": neg_log_likelihood,
            "bpb": bpb, 
            "bpc": bpc, 
            "bpw": bpw
        }
print("IntrinsicEvaluator class defined.")

IntrinsicEvaluator class defined.


Standardized Data Loading & Preprocessing

In [6]:
diverse_sentences_path = "/kaggle/input/datasets/shanilpraveen/intrinsic/diverse_sentences.csv"
sinhala_mixed_path = "/kaggle/input/datasets/shanilpraveen/intrinsic/sinhala_mixed_dataset.csv"

try:
    diverse_df = pd.read_csv(diverse_sentences_path).dropna()
    mixed_df = pd.read_csv(sinhala_mixed_path).dropna()
    print(f"Loaded Diverse Sentences: {len(diverse_df)} parallel pairs.")
    print(f"Loaded Mixed Script: {len(mixed_df)} sentences.")
except Exception as e:
    print(f"Data loading failed. Check your file paths. Error: {e}")

Loaded Diverse Sentences: 500 parallel pairs.
Loaded Mixed Script: 500 sentences.


Model & Tokenizer Initialization

In [8]:
import os

MODEL_PATH = "/kaggle/input/models/manojkumarcs28/cerebras-gpt-1-3b/pytorch/default/1/Cerebras-GPT-1.3B"

print(f"Loading Cerebras-GPT-1.3B from: {MODEL_PATH}")
print("Files:", os.listdir(MODEL_PATH))

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model natively in fp16 across both GPUs
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    local_files_only=True
)
model.eval()

# Initialize custom evaluator
evaluator = IntrinsicEvaluator(model, tokenizer)

print(f"Model loaded.")
print(f"Precision: {model.dtype} | Device Map: {model.hf_device_map}")

Loading Cerebras-GPT-1.3B from: /kaggle/input/models/manojkumarcs28/cerebras-gpt-1-3b/pytorch/default/1/Cerebras-GPT-1.3B
Files: ['config.json', 'merges.txt', 'README.md', 'vocab.json', 'gitattributes', 'pytorch_model.bin']


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: /kaggle/input/models/manojkumarcs28/cerebras-gpt-1-3b/pytorch/default/1/Cerebras-GPT-1.3B
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.
Precision: torch.float16 | Device Map: {'transformer.wte': 0, 'lm_head': 0, 'transformer.wpe': 0, 'transformer.drop': 0, 'transformer.h.0': 0, 'transformer.h.1': 0, 'transformer.h.2': 0, 'transformer.h.3': 0, 'transformer.h.4': 0, 'transformer.h.5': 0, 'transformer.h.6': 0, 'transformer.h.7': 0, 'transformer.h.8': 0, 'transformer.h.9': 0, 'transformer.h.10': 0, 'transformer.h.11': 1, 'transformer.h.12': 1, 'transformer.h.13': 1, 'transformer.h.14': 1, 'transformer.h.15': 1, 'transformer.h.16': 1, 'transformer.h.17': 1, 'transformer.h.18': 1, 'transformer.h.19': 1, 'transformer.h.20': 1, 'transformer.h.21': 1, 'transformer.h.22': 1, 'transformer.h.23': 1, 'transformer.ln_f': 1}


Pilot Run

In [9]:
import csv

pilot_output_path = 'pilot_intrinsic_results.csv'
print("Starting Pilot Run (First 20 items)...")

# Open CSV in append mode to protect against runtime crashes
with open(pilot_output_path, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    # Write standardized header
    writer.writerow([
        'item_index', 
        'unicode_nll', 'unicode_bpb', 'unicode_bpc', 'unicode_bpw',
        'romanized_nll', 'romanized_bpb', 'romanized_bpc', 'romanized_bpw'
    ])
    
    # Slice the first 20 rows of our loaded Kaggle dataframe
    pilot_sample = diverse_df.head(20)
    
    for index, row in pilot_sample.iterrows():
        # 1. Evaluate Native Unicode
        uni_metrics = evaluator.evaluate_sequence(row['sinhala_unicode'])
        
        # 2. Evaluate Romanized (Singlish)
        rom_metrics = evaluator.evaluate_sequence(row['sinhala_romanized'])
        
        # Log to disk
        writer.writerow([
            index, 
            uni_metrics['nll'], uni_metrics['bpb'], uni_metrics['bpc'], uni_metrics['bpw'],
            rom_metrics['nll'], rom_metrics['bpb'], rom_metrics['bpc'], rom_metrics['bpw']
        ])
        
        if (index + 1) % 5 == 0:
            print(f"Processed {index + 1}/20 parallel pairs...")

print(f"\n Pilot run complete. Results saved to {pilot_output_path}")

# Display verification preview
preview_df = pd.read_csv(pilot_output_path)
print("\nFirst 3 rows of calculated metrics:")
display(preview_df.head(3))

Starting Pilot Run (First 20 items)...


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Processed 5/20 parallel pairs...
Processed 10/20 parallel pairs...
Processed 15/20 parallel pairs...
Processed 20/20 parallel pairs...

 Pilot run complete. Results saved to pilot_intrinsic_results.csv

First 3 rows of calculated metrics:


,item_index,unicode_nll,unicode_bpb,unicode_bpc,unicode_bpw,romanized_nll,romanized_bpb,romanized_bpc,romanized_bpw
0,0,74.630705,1.380376,3.588978,17.944891,105.229340,3.995101,3.995101,25.302308
1,1,57.343391,1.253470,3.447043,20.682257,78.418572,4.040503,4.040503,28.283521
2,2,72.640600,1.262629,3.380588,17.466372,94.352266,3.781154,3.781154,22.686924


Full Intrinsic Evaluation

In [10]:
import csv
import pandas as pd
from tqdm import tqdm

# Evaluate Parallel Diverse Sentences (Unicode vs Romanized)
full_diverse_output = 'Cerebras-GPT-1.3B_diverse_intrinsic.csv'
print(f"Starting full parallel evaluation on {len(diverse_df)} items...")

with open(full_diverse_output, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow([
        'item_index',
        'unicode_nll', 'unicode_bpb', 'unicode_bpc', 'unicode_bpw',
        'romanized_nll', 'romanized_bpb', 'romanized_bpc', 'romanized_bpw'
    ])

    for index, row in tqdm(diverse_df.iterrows(), total=len(diverse_df), desc="Processing Parallel Pairs"):
        uni_metrics = evaluator.evaluate_sequence(row['sinhala_unicode'])
        rom_metrics = evaluator.evaluate_sequence(row['sinhala_romanized'])

        writer.writerow([
            index,
            uni_metrics['nll'], uni_metrics['bpb'], uni_metrics['bpc'], uni_metrics['bpw'],
            rom_metrics['nll'], rom_metrics['bpb'], rom_metrics['bpc'], rom_metrics['bpw']
        ])

print(f"Diverse evaluation complete. Saved to {full_diverse_output}")

# Evaluate Authentic Mixed-Script Sentences
full_mixed_output = 'Cerebras-GPT-1.3B_mixed_intrinsic.csv'
mixed_text_col = 'text' if 'text' in mixed_df.columns else mixed_df.columns[0]
print(f"\nStarting full mixed-script evaluation on {len(mixed_df)} items (column: '{mixed_text_col}')...")

with open(full_mixed_output, mode='w', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow(['item_index', 'mixed_nll', 'mixed_bpb', 'mixed_bpc', 'mixed_bpw'])

    for index, row in tqdm(mixed_df.iterrows(), total=len(mixed_df), desc="Processing Mixed Script"):
        mix_metrics = evaluator.evaluate_sequence(row[mixed_text_col])
        writer.writerow([
            index,
            mix_metrics['nll'], mix_metrics['bpb'], mix_metrics['bpc'], mix_metrics['bpw']
        ])

print(f"Mixed-script evaluation complete. Saved to {full_mixed_output}")

# Aggregate and Display Unicode vs Romanized Summary
def calculate_full_aggregates(csv_path, model_label="Cerebras-GPT-1.3B"):
    df = pd.read_csv(csv_path)

    summary = {
        'Metric': ['Bits-Per-Byte (BPB)', 'Bits-Per-Character (BPC)', 'Bits-Per-Word (BPW)'],
        'Unicode (Native)': [
            df['unicode_bpb'].mean(),
            df['unicode_bpc'].mean(),
            df['unicode_bpw'].mean()
        ],
        'Romanized (Singlish)': [
            df['romanized_bpb'].mean(),
            df['romanized_bpc'].mean(),
            df['romanized_bpw'].mean()
        ]
    }

    results_df = pd.DataFrame(summary)
    results_df['Degradation Factor'] = results_df['Romanized (Singlish)'] / results_df['Unicode (Native)']

    print("==========================================================")
    print(f"      {model_label} INTRINSIC EVALUATION SUMMARY")
    print("==========================================================")
    print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("==========================================================")

    styled = results_df.style.format({
        'Unicode (Native)': '{:.4f}',
        'Romanized (Singlish)': '{:.4f}',
        'Degradation Factor': '{:.2f}×'
    }).set_caption(f"{model_label} — Unicode vs Romanized Sinhala").background_gradient(
        subset=['Degradation Factor'], cmap='Reds'
    )
    display(styled)

    return results_df

full_summary = calculate_full_aggregates(full_diverse_output)

Starting full parallel evaluation on 500 items...


Processing Parallel Pairs: 100%|██████████| 500/500 [00:25<00:00, 19.81it/s]


Diverse evaluation complete. Saved to Cerebras-GPT-1.3B_diverse_intrinsic.csv

Starting full mixed-script evaluation on 500 items (column: 'text')...


Processing Mixed Script: 100%|██████████| 500/500 [00:12<00:00, 38.75it/s]


Mixed-script evaluation complete. Saved to Cerebras-GPT-1.3B_mixed_intrinsic.csv
      Cerebras-GPT-1.3B INTRINSIC EVALUATION SUMMARY
                  Metric  Unicode (Native)  Romanized (Singlish)  Degradation Factor
     Bits-Per-Byte (BPB)            1.5136                3.7264              2.4620
Bits-Per-Character (BPC)            4.0023                3.7350              0.9332
     Bits-Per-Word (BPW)           21.4649               23.0476              1.0737


,Metric,Unicode (Native),Romanized (Singlish),Degradation Factor
0,Bits-Per-Byte (BPB),1.5136,3.7264,2.46×
1,Bits-Per-Character (BPC),4.0023,3.7350,0.93×
2,Bits-Per-Word (BPW),21.4649,23.0476,1.07×


In [11]:
def calculate_mixed_aggregates(csv_path, model_label="Cerebras-GPT-1.3B"):
    df = pd.read_csv(csv_path)

    summary = {
        'Metric': ['Bits-Per-Byte (BPB)', 'Bits-Per-Character (BPC)', 'Bits-Per-Word (BPW)'],
        'Mixed-Script': [
            df['mixed_bpb'].mean(),
            df['mixed_bpc'].mean(),
            df['mixed_bpw'].mean()
        ],
        'Std Dev': [
            df['mixed_bpb'].std(),
            df['mixed_bpc'].std(),
            df['mixed_bpw'].std()
        ]
    }

    results_df = pd.DataFrame(summary)

    print("==========================================================")
    print(f"      {model_label} MIXED-SCRIPT INTRINSIC SUMMARY")
    print(f"      ({len(df)} sentences)")
    print("==========================================================")
    print(results_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("==========================================================")

    styled = results_df.style.format({
        'Mixed-Script': '{:.4f}',
        'Std Dev': '{:.4f}'
    }).set_caption(f"{model_label} — Mixed-Script Sinhala")
    display(styled)

    return results_df

mixed_summary = calculate_mixed_aggregates(full_mixed_output)

      Cerebras-GPT-1.3B MIXED-SCRIPT INTRINSIC SUMMARY
      (500 sentences)
                  Metric  Mixed-Script  Std Dev
     Bits-Per-Byte (BPB)        2.1597   0.5392
Bits-Per-Character (BPC)        4.1128   0.4269
     Bits-Per-Word (BPW)       25.0708   4.2866


,Metric,Mixed-Script,Std Dev
0,Bits-Per-Byte (BPB),2.1597,0.5392
1,Bits-Per-Character (BPC),4.1128,0.4269
2,Bits-Per-Word (BPW),25.0708,4.2866
